# L13a: Model Free Reinforcement Learning
In this lecture, we will explore the fundamentals of Model Free Reinforcement Learning (MFRL), a branch of machine learning focused on how agents should take actions in an environment to maximize cumulative reward over time. 

> __Learning Objectives__
> By the end of this module, you will be able to define and demonstrate mastery of the following key concepts:
>
> * __Exploration vs. Exploitation__: In reinforcement learning, the exploration vs. exploitation trade‐off forces an agent to balance trying new actions to discover potentially better rewards (exploration) against leveraging its current knowledge to maximize immediate payoff (exploitation). Striking the right balance is crucial for learning an optimal policy that performs well both now and in the long run.
> * __Value Iteration (Review)__: A dynamic programming algorithm that computes the optimal value function and policy for Markov decision processes when the transition model and reward function are known, serving as the foundation for understanding model-based reinforcement learning approaches.
> * __Q-learning__: A model-free, off-policy algorithm that learns the optimal action-value function by iteratively updating its value estimates using observed rewards and the best available future estimates. Q-learning converges to the optimal policy under conditions of proper learning rate decay and infinite exploration, without requiring any model of the environment.


At its core, RL is about learning from doing: an agent observes the state of its environment, takes actions, i.e., makes decisions, receives rewards, and updates its knowledge to improve future decision-making. 
Let’s get started!

___

## Examples
Today, we will use the following examples to illustrate key concepts:

> [▶ Solve the lava-world navigation problem using Q-learning](CHEME-5800-L13a-Example-LavaWorldProblem-Q-Learning-Fall-2025.ipynb). In this example, we'll revisit the lava-world navigation problem (which we previously solved using Value Iteration) and implement the Q-learning algorithm to help an agent learn to navigate safely to its goal while avoiding hazards. 

___

## Reinforcement Learning Problem
Suppose we have an agent that can be in a state $s \in \mathcal{S}$ and can take an action $a \in \mathcal{A}$. After taking action $a$ in state $s$, the agent receives a reward $r$. But how does the agent learn to choose the best possible action in each state to maximize its cumulative reward over time?

<div>
    <center>
        <img src="figs/Fig-Schematic-RL.svg" width="580"/>
    </center>
</div>

In reinforcement learning, an agent interacts with an environment by observing its current state $s \in \mathcal{S}$, selecting an action $a \in \mathcal{A}$, and receiving a reward that influences its future decisions. We have explored different approaches to this problem:

* __Multiplicative weights__ approaches the probability of selecting an action based on past performance, but they do so in a principled way that guarantees the algorithm performs nearly as well as the best fixed action in hindsight—even in changing environments, i.e., it minimizes regret.
* __Bandit algorithms__ operate in stateless environments. On each round, they explore different actions to estimate their rewards and adapt their action-selection strategy based on the outcomes.
* __Q-learning__ is a value-based method that estimates the long-term value (utility, satisfaction, happiness, etc) of each state-action pair, enabling the agent to learn optimal behavior in environments with temporal and sequential dynamics.

These approaches highlight different strategies for learning from interaction, but they all must balance a fundamental challenge in reinforcement learning: the tradeoff between exploring new actions to gather information and exploiting known actions to maximize reward.

___

## Review: Markov Decision Processes (MDPs)
A Markov decision process (MDP) is a formal model of the agent-environment interaction shown above, for settings where outcomes are partly random and partly under the agent's control. An MDP is defined by the tuple $\left(\mathcal{S}, \mathcal{A}, R(s,a), T\left(s^{\prime}\,|\,s,a\right), \gamma\right)$:

* __States__: The state space $\mathcal{S}$ is the set of all states $s\in\mathcal{S}$ the system can occupy. For example, we could define a state space of investor moods $\mathcal{S} \equiv \left\{\text{bullish},\text{neutral},\text{bearish}\right\}$.
* __Actions__: The action space $\mathcal{A}$ is the set of all actions $a\in\mathcal{A}$ available to the agent, where $\mathcal{A}_{s} \subseteq \mathcal{A}$ is the subset accessible from state $s$. In the investor example, $\mathcal{A} \equiv \left\{\text{buy},\text{hold},\text{sell}\right\}$.
* __Reward__: The reward $R(s,a)$ is received for taking action $a$ in state $s$. For example, this could be the proceeds (or losses) from selling shares of asset `XYZ`.
* __Transitions__: The transition model $T\left(s^{\prime}\,|\,s,a\right) = P\left(s_{t+1} = s^{\prime}\,|\,s_{t}=s, a_{t}=a\right)$ is the probability that action $a$ in state $s$ at time $t$ leads to state $s^{\prime}$ at time $t+1$. It has the Markov property but conditions on both the current state $s$ and the action $a$.
* __Discount__: The discount factor $0<\gamma<1$ weighs future utility against immediate utility and is a hyperparameter of the model.

A policy function $\pi:\mathcal{S}\rightarrow\mathcal{A}$ maps each state $s\in\mathcal{S}$ to an action $a\in\mathcal{A}$. Our goal is to find an optimal policy $\pi^{*}(s)$ that gives the best possible decisions. The next two sections develop two ways to do this: value iteration, which assumes the reward $R(s,a)$ and transition model $T\left(s^{\prime}\,|\,s,a\right)$ are known, and Q-learning, which does not.
___

## Value Functions, Value Iteration, and Policy Functions
With the MDP framework in place, we can compute an optimal policy. We start with value iteration, which applies when the reward $R(s,a)$ and transition model $T\left(s^{\prime}\,|\,s,a\right)$ are known.

Value iteration is a dynamic programming algorithm that computes the __optimal value function__ $U^{*}(s)$ (utility, satisfaction, etc) by iteratively applying the __Bellman backup operation__:

$$
\begin{equation*}
U_{k+1}(s) = \max_{a\in\mathcal{A}}\left(\underbrace{R(s,a)}_{\text{= now}} + 
\gamma\;\overbrace{\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U}_{k}(s^{\prime})}^{\text{= future}}\right)
\end{equation*}
$$

As $k \to \infty$, the value function is __guaranteed to converge__ for $0\leq\gamma<1$ such that $U_k(s) \to U^{*}(s)$. The optimal value function represents the maximum expected cumulative discounted reward achievable from each state under the best possible policy. Let's develop the value iteration algorithm for computing the optimal value function $U^{*}(s)$ and policy $\pi^{*}(s)$.

### Algorithm: Value Iteration

__Initialize__: Given an MDP with state space $\mathcal{S}$, action space $\mathcal{A}$, reward function $R(s,a)$, transition model $T\left(s^{\prime}\,|\,s,a\right)$, discount factor $\gamma$, tolerance parameter $\epsilon$, and maximum number of iterations $T$. Initialize the iteration counter $k\gets 0$, the initial value function $U_{0}(s) \gets 0$ for all $s \in \mathcal{S}$, and $\texttt{converged}\gets\texttt{false}$.

While $\texttt{converged}$ is $\texttt{false}$ __do__:
1. For each state $s \in \mathcal{S}$, compute the updated value:
   $$U_{k+1}(s) \gets \max_{a\in\mathcal{A}}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U}_{k}(s^{\prime})\right)$$
2. Check for convergence:
    - If $\max_{s\in\mathcal{S}} \left|U_{k+1}(s) - U_{k}(s)\right| \leq \epsilon$, then set $\texttt{converged}\gets\texttt{true}$ and $U^{*}\gets{U}_{k+1}$.
    - If $\max_{s\in\mathcal{S}} \left|U_{k+1}(s) - U_{k}(s)\right| > \epsilon$, update $k\gets{k+1}$ and $U_{k}\gets{U}_{k+1}$.
3. Update the $\texttt{converged}$ flag:
    - If $k\geq{T}$, then set $\texttt{converged}\gets\texttt{true}$ and $U^{*}\gets{U}_{k+1}$. Notify the caller that the maximum iteration limit was reached without convergence.

__Extract Policy__: For each state $s \in \mathcal{S}$, compute:
$$\pi^{*}(s) \gets \arg\max_{a\in\mathcal{A}}\left(\underbrace{R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U^{*}}(s^{\prime})}_{\text{state-action-value } Q(s,a)}\right)$$

### Convergence

Because the Bellman backup is a contraction operator, repeated backups are guaranteed to converge to a unique fixed point, so $U_{k}(s) \to U^{*}(s)$ as $k\to\infty$ for $0\leq\gamma<1$. The number of iterations needed to reach accuracy $\epsilon$ has the upper bound:

$$
\begin{equation*}
T \sim \left(\frac{1}{1-\gamma}\right)\cdot\ln\left(\frac{1}{\epsilon}\right)
\end{equation*}
$$

As the discount factor $\gamma$ approaches `1`, the number of iterations required for convergence grows, which is an important practical consideration.

* __Long-term vs. short-term focus:__ A discount factor $\gamma$ near 1 weights future rewards heavily, giving more globally optimal decisions but slower convergence. A smaller $\gamma$ prioritizes immediate rewards, speeding convergence at the risk of suboptimal long-run performance.
* __Choosing $\gamma$:__ Select $\gamma$ from the effective planning horizon $H$ using the guideline $\gamma \approx 1 - 1/H$. For outcomes 100 steps ahead, use $\gamma\approx0.99$; for 10 steps, use $\gamma\approx0.9$.

### Computational Complexity

Value iteration requires $O(|\mathcal{S}|^2 \cdot |\mathcal{A}|)$ operations per iteration. For each of the $|\mathcal{S}|$ states, we must evaluate $|\mathcal{A}|$ actions, and each action evaluation requires summing over $|\mathcal{S}|$ possible next states.

This complexity makes value iteration tractable for problems with thousands of states but challenging for very large discrete state spaces. In such cases, function approximation or sampling-based methods become necessary.
___


<div>
    <center>
        <img src="figs/Fig-Q-Schematic.svg" width="580"/>
    </center>
</div>

## Q-Learning
In Markov decision processes (MDPs), the __state-action-value function__ $Q(s,a)$ (also called the Q-function) represents the expected cumulative discounted reward of taking action $a$ in state $s$ and following the optimal policy thereafter. We have a model of the environment defined by the transition probabilities $T\left(s^{\prime}\,|\,s,a\right)$ and the reward function $R(s,a)$. 

However, in many real-world scenarios, we do not have access to this model. Instead, we can learn the optimal policy directly from experience using __Q-learning__, a model-free reinforcement learning algorithm.

Q-learning iteratively estimates the state action-value function $Q(s, a)$ by conducting repeated experiments $t=1,2,\ldots$ in the world $\mathcal{W}$. 
In each experiment, an agent in state $s\in\mathcal{S}$ takes action $a\in\mathcal{A}$, receives a reward $r$, and (potentially) transitions to a new state $s^{\prime}$. After each experiment $t$, the agent updates its estimate of $Q(s, a)$ using the update rule:
$$
\begin{equation*}
Q_{t+1}(s,a)\leftarrow{\underbrace{Q_{t}(s,a)}_{\text{old value}}}+\alpha_{t}\cdot\underbrace{\left(r+\gamma\cdot\max_{a^{\prime}\in\mathcal{A}}Q_{t}(s^{\prime},a^{\prime}) - Q_{t}(s,a)\right)}_{\text{new value}}\quad{t = 1,2,3,\ldots}
\end{equation*}
$$
where $0<\alpha_{t} <{1}$ is the learning rate parameter at time $t$, and $0<\gamma<{1}$ is the discount factor. 
We estimate the policy function $\pi:\mathcal{S}\rightarrow\mathcal{A}$ by selecting the action $a$ that maximizes $Q(s,a)$ at each state $s$:
$$
\begin{equation*}
\pi(s) = \arg\max_{a\in\mathcal{A}}Q(s,a)
\end{equation*}
$$

### Algorithm
Initialize $Q(s,a)$ arbitrarily for all $s\in\mathcal{S}$, and $a\in\mathcal{A}$.
Set the hyperparameters: learning rate $\alpha_{t}$, the discount factor $\gamma$, the exploration rate $\epsilon_{t}$,the maximum number of iterations $\texttt{maxiter}$, and the convergence tolerance $\delta$. Set the $\texttt{converged}\gets\texttt{false}$. 

For $s\in\mathcal{S}$
1. Initialize the trial counter $t\gets{1}$
2. While $\texttt{converged} $ is $\texttt{false}$ __do__:
    1. Roll a random number $p\in[0,1]$. Compute $\epsilon_{t}={t^{-1/3}}\cdot\left(K\cdot\log(t)\right)^{1/3}$ where $K=|\mathcal{A}|$ is the number of actions.
    2. If $p\leq\epsilon_{t}$, choose a random (uniform) action $a_{t}\in\mathcal{A}$. Otherwise, choose a greedy action $a_{t} = \text{arg}\max_{a\in\mathcal{A}}{Q_{t}(s,a)}$.
    3. Take action $a_{t}$, observe the reward $r$ from the __world__ and transition to the next state $s^{\prime}$.
    4. Update the state-action-value function: $Q_{t+1}(s,a)\leftarrow{Q_{t}(s,a)}+\alpha_{t}\cdot\underbrace{\left(r+\gamma\cdot\overbrace{\max_{a^{\prime}\in\mathcal{A}}Q_{t}(s^{\prime},a^{\prime})}^{\text{one-step lookahead}} - Q_{t}(s,a)\right)}_{\text{new information}}$.
    5. Update the state $s\leftarrow{s^{\prime}}$, the learning rate $\alpha_{t+1}\leftarrow\alpha_{t}$ and the counter $t\leftarrow{t+1}$
    6. Convergence check: If the $Q(s,a)$ has bounded change $\lVert{Q_{t+1}(s,a) - Q_{t}(s,a)}\rVert\leq\delta$, then the algorithm has converged. Set $\texttt{converged}\gets\texttt{true}$.
    7. Otherwise: if $t\geq\texttt{maxiter}$, then set $\texttt{converged}\gets\texttt{true}$ and notify the caller that the maximum iteration limit was reached without convergence. Proceed to next state.
    8. Otherwise: continue to the next iteration.
3. End While
4. End For

### Convergence
Q-learning converges to the optimal state-action-value function $Q_{t}(s,a)\to{Q^{*}(s,a)}$ with probability 1 under the following conditions (assuming the Markov property holds for the environment):
* __Finite MDP and discounting__: The state-action space $\mathcal{S}\times\mathcal{A}$ must be finite, and the discount factor must satisfy $0\leq\gamma<1$.
* __Learning rate decay__: The learning rate $\alpha_{t}$ must satisfy $\sum_{t=0}^\infty \alpha_t(s, a) = \infty$ and $\sum_{t=0}^\infty \alpha_t^2(s, a) < \infty$ for all state-action pairs, ensuring sufficient initial updates while stabilizing over time. A common choice is $\alpha_{t} = 1/(1+t)^{\kappa}$ with $1/2<\kappa\leq{1}$.
* __Infinite exploration__: All state-action pairs must be visited (and updated) infinitely often. This condition holds for $\epsilon$-greedy policies with persistent exploration, i.e., $\epsilon_{t} > 0\,\,\forall{t}$.

The infinite-exploration condition is the difficult one in practice. We therefore run the algorithm for multiple __episodes__, initializing $Q(s,a)$ with the output of the previous episode so the agent learns from accumulated experience. Repeated episodes ensure every state-action pair is visited often enough for the estimates to converge.

### Caveats & Practical Notes
* __Wandering away__: If the dynamics never return to the current state $s$, that state may be updated only once per sweep and fail the infinitely-often requirement. Ensure each $(s,a)$ pair is updated repeatedly, for example by resetting to $s$ or bounding the sweep length.
* __Exploration vs. exploitation trade-off__: If $\epsilon_{t}$ decays too fast, the agent stops exploring before visiting all pairs enough times; if it decays too slowly, learning is noisy. Tuning the decay schedule is critical.
* __Rate of convergence__: Convergence is almost sure, but its speed is not guaranteed. Asynchronous sampling can be slow in large state and action spaces, so many implementations use __replay buffers__ or fall back to value iteration when a model is available.

Let's look at an example to see how Q-learning works in practice.

> __Example:__
> 
> [▶ Solve the lava-world navigation problem using Q-learning](CHEME-5800-L13a-Example-LavaWorldProblem-Q-Learning-Fall-2025.ipynb). In this example, we'll revisit the lava-world navigation problem (which we previously solved using Value Iteration) and implement the Q-learning algorithm to help an agent learn to navigate safely to its goal while avoiding hazards. 

___

## Summary
In this lecture, we explored reinforcement learning fundamentals, focusing on how agents learn to make decisions through interaction with their environment.

> __Key Takeaways:__
>
> * **Exploration vs. exploitation trade-off:** Agents must balance trying new actions to discover better strategies against using known actions to maximize immediate rewards. This fundamental challenge appears across reinforcement learning approaches from bandit algorithms to value-based methods like Q-learning.
> * **Value iteration for model-based RL:** When the transition model and rewards are known, value iteration computes optimal policies by iteratively applying the Bellman backup operation until convergence, requiring $O(|\mathcal{S}|^2 \cdot |\mathcal{A}|)$ operations per iteration and complete knowledge of the environment dynamics.
> * **Q-learning for model-free RL:** Q-learning discovers optimal policies through direct interaction with the environment without requiring a model. Using epsilon-greedy exploration and temporal difference updates, it converges to optimality under conditions of proper learning rate decay and infinite exploration, making it applicable to real-world scenarios where environmental models are unavailable.

Reinforcement learning enables agents to learn optimal behavior through trial and error in both known and unknown environments, with different algorithmic approaches suited to different levels of environmental knowledge.
___

## References
General references for further reading on reinforcement learning and Q-learning:

1. **Bellman, R.** (1957). *Dynamic Programming*. Princeton University Press.
2. **Puterman, M. L.** (1994). *Markov Decision Processes: Discrete Stochastic Dynamic Programming*. John Wiley & Sons.
3. **Watkins, C. J. C. H., & Dayan, P.** (1992). Q-learning. *Machine Learning*, 8(3-4), 279-292.
4. **Watkins, C. J. C. H.** (1989). *Learning from Delayed Rewards*. PhD thesis, Cambridge University.
5. **Sutton, R. S., & Barto, A. G.** (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.
6. **Bertsekas, D. P., & Tsitsiklis, J. N.** (1996). *Neuro-Dynamic Programming*. Athena Scientific.
7. **Jaakkola, T., Jordan, M. I., & Singh, S. P.** (1994). On the convergence of stochastic iterative dynamic programming algorithms. *Neural Computation*, 6(6), 1185-1201.

___